<a href="https://colab.research.google.com/github/ANitkin8/developersinstitute/blob/main/airplane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
df = pd.read_csv("Airplane_Crashes.csv", encoding="cp1252")
df_clean = df.copy()
row_count = len(df)
missing_counts = df.isnull().sum()
missing_values =[r"^\s*$", r"^(?i)(null|n/a|na|none)$"]
df_clean = df_clean.replace(missing_values, np.nan,)
numeric_cols = ["Aboard", "Aboard Passangers", "Aboard Crew",  "Fatalities" "Fatalities Passangers", "Fatalities Crew", "Ground" ]
for col in numeric_cols:
  if col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
#fill blank data of text columns
obj_cols = df_clean.select_dtypes(include=["object", "str"]).columns
df_clean[obj_cols] = df_clean[obj_cols].fillna("Unknown")
#convert the date
df_clean["Date"] = pd.to_datetime(df_clean["Date"], format="%m/%d/%Y")
df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.month
#explore survivors, fatalities
df_clean["Survivors"] = df_clean["Aboard"]-df_clean["Fatalities"]
total_fatalities = df_clean["Fatalities"].sum()
total_survivors = df_clean["Survivors"].sum()
print(f"Crashes = {row_count} Total fatalities: {total_fatalities} Total Survivors: {total_survivors}")
#create a table showing crashes by decade
df_clean["Decade"] = (df_clean["Date"].dt.year // 10) * 10
decade_summary = (
    df_clean.groupby("Decade")
    .agg(
        total_crashes=("Date", "count"),
        total_fatalities=("Fatalities", "sum"),
        avg_fatalities=("Fatalities", "mean"),
    )
    .sort_index()
)
decade_summary["Decade_Label"] = [f"{int(d)}s" for d in decade_summary.index]

print(decade_summary.round(2))
#find the survival rate
df_clean["Survival_Rate"] = (df_clean["Survivors"] / df_clean["Aboard"]) * 100
#calc  mean, median, and standard deviation of fatalities and survival rates
stats_summary = df_clean [["Fatalities", "Survival_Rate"]].agg(["mean", "std","median"])
print(stats_summary.round(2))
#create visualization
import matplotlib.pyplot as plt
labels = [f"{int(d)}s" for d in decade_summary.index]
x = np.arange(len(labels))
width = 0.38

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), sharex=True)
ax1_twin = ax1.twinx()

ax1.bar(x - width / 2, decade_summary["total_crashes"], width=width, label="Total Crashes")
ax1_twin.bar(
    x + width / 2,
    decade_summary["total_fatalities"],
    width=width,
    color="crimson",
    alpha=0.85,
    label="Total Fatalities",
)
ax1.set_ylabel("Total Crashes")
ax1_twin.set_ylabel("Total Fatalities", color="crimson")
ax1.set_title("Crashes and Fatalities by Decade")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax1_twin.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")

ax2.bar(x, decade_summary["avg_fatalities"], width=0.5, color="darkorange", label="Avg Fatalities / Crash")
ax2.set_ylabel("Avg Fatalities / Crash")
ax2.set_xlabel("Decade")
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

It was very interesting reading this data. Some of the things that stood out were that while the number of crashes started to decrease from the 1950s through the 80s, there was a jump in the 1990s. The 1970s had by far the most fatalities. It was also noteworthy seeing the number of crashes drop greatly in the 2000s and the 2010s. Perhaps this is a result of improved pilot training. It would be worth exploring further to see why the trends happend the way they did.